# AsyncFlow — MMc Theory vs Simulation (Guided Notebook)

This notebook shows how to:

1. Make imports work inside a notebook (src-layout or package install)
2. Build a **multi-server** scenario compatible with **M/M/c** assumptions
3. Run the simulation and collect results
4. Compare theory vs observed KPIs (pretty-printed table)
5. Plot the standard dashboards (latency, throughput, server time series)




In [9]:
import sys, importlib


for m in list(sys.modules):
    if m.startswith("asyncflow"):
        del sys.modules[m]


from asyncflow import AsyncFlow, SimulationRunner
from asyncflow.analysis import MMc, ResultsAnalyzer
from asyncflow.components import (
    Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
)
from asyncflow.settings import SimulationSettings

import simpy

In [10]:
import matplotlib.pyplot as plt
import simpy

# Public AsyncFlow API
from asyncflow import AsyncFlow, SimulationRunner, Sweep
from asyncflow.components import Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
from asyncflow.settings import SimulationSettings
from asyncflow.analysis import  ResultsAnalyzer, SweepAnalyzer, MMc
from asyncflow.enums import Distribution

print("Imports OK.")

Imports OK.


## 1) Build an M/M/c split-friendly scenario

* **Multiple identical servers with exponential CPU service**
  Topology includes **\$c \geq 2\$ identical servers**, each exposing exactly **one endpoint** with exactly **one CPU-bound step**.
  Service times follow an **Exponential** distribution with mean \$E\[S]\$ (service rate \$\mu = 1/E\[S]\$). No RAM/IO steps are included in the pipeline.

* **Load balancer with FCFS dispatch**

* **“Poisson arrivals” via the generator**
  
  

---

```mermaid
graph LR;
    rqs1["<b>RqsGenerator</b><br/>id: rqs-1"]
    client1["<b>Client</b><br/>id: client-1"]
    lb1["<b>LoadBalancer</b><br/>id: lb-1<br/>Policy: round_robin"]
    app1["<b>Server</b><br/>id: app-1<br/>Endpoint: /api"]
    app2["<b>Server</b><br/>id: app-2<br/>Endpoint: /api"]

    rqs1 -- "Edge: gen-client<br/>Latency: 0.0001" --> client1;
    client1 -- "Request<br/>Edge: client-lb<br/>Latency: 0.0001" --> lb1;
    lb1 -- "Dispatch<br/>Edge: lb-app1<br/>Latency: 0.0001" --> app1;
    lb1 -- "Dispatch<br/>Edge: lb-app2<br/>Latency: 0.0001" --> app2;
    app1 -- "Response<br/>Edge: app1-client<br/>Latency: 0.0001" --> client1;
    app2 -- "Response<br/>Edge: app2-client<br/>Latency: 0.0001" --> client1;
```

---



In [ ]:
def build_payload():
    generator = ArrivalsGenerator(
        id="rqs-1",
        lambda_rps=30,
        model=Distribution.POISSON
    )

    client = Client(id="client-1")

    endpoint = Endpoint(
        endpoint_name="/api",
        probability=1.0,
        steps=[
            {
                "kind": "initial_parsing",
                "step_operation": {
                    "cpu_time": {"mean": 0.01, "distribution": "exponential"},
                },
            },
        ],
    )

    srv1 = Server(
        id="srv-1",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    srv2 = Server(
        id="srv-2",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    
    srv3 = Server(
        id="srv-3",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )

    lb = LoadBalancer(
        id="lb-1",
        algorithms="fcfs",  
        server_covered={"srv-1", "srv-2", "srv-3"},
    )

    edges = [
        LinkEdge(id="gen-client",  source="rqs-1",  target="client-1",),
        LinkEdge(id="client-lb",   source="client-1", target="lb-1",  ),
        LinkEdge(id="lb-srv1",     source="lb-1",   target="srv-1",   ),
        LinkEdge(id="lb-srv2",     source="lb-1",   target="srv-2",   ),
        LinkEdge(id="lb-srv3",     source="lb-1",   target="srv-3",   ),
        LinkEdge(id="srv1-client", source="srv-1",  target="client-1",),
        LinkEdge(id="srv2-client", source="srv-2",  target="client-1",),
        LinkEdge(id="srv3-client", source="srv-3",  target="client-1",),
    ]

    settings = SimulationSettings(
        total_simulation_time=2400,
        sample_period_s=0.05,
    )

    payload = (
        AsyncFlow()
        .add_arrivals_generator(generator)
        .add_client(client)
        .add_servers(srv1, srv2, srv3)
        .add_load_balancer(lb)
        .add_edges(*edges)
        .add_simulation_settings(settings)
    ).build_payload()

    return payload


## 2) Run the simulation

In [12]:
payload = build_payload()
env = simpy.Environment()
runner = SimulationRunner(env=env, simulation_input=payload)
results: ResultsAnalyzer = runner.run()
print("Done.")

ValidationError: 1 validation error for TopologyNodes
  Value error, Load balancer 'lb-1' references unknown server 'srv-3'. Define it under 'servers' or remove it from 'server_covered'. [type=value_error, input_value={'servers': [Server(id='s..., ram_per_process=None)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error